In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets scikit-learn matplotlib kagglehub pynvml

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from PIL import Image

from tqdm import tqdm

from datasets import Dataset

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report,

    roc_curve,

    auc
)

from transformers import (

    InstructBlipProcessor,

    InstructBlipForConditionalGeneration,

    BitsAndBytesConfig,

    TrainingArguments,

    Trainer
)

from peft import (

    LoraConfig,

    get_peft_model,

    prepare_model_for_kbit_training
)

In [ ]:
path = kagglehub.dataset_download(
    "nrizwan/mami-dataset"
)

print(path)

In [ ]:
TRAIN_CSV = os.path.join(

    path,

    "MAMI_Dataset",

    "data",

    "train.tsv"
)

DEV_CSV = os.path.join(

    path,

    "MAMI_Dataset",

    "data",

    "validation.tsv"
)

TEST_CSV = os.path.join(

    path,

    "MAMI_Dataset",

    "data",

    "test.tsv"
)

TRAIN_IMAGE_DIR = os.path.join(

    path,

    "MAMI_Dataset",

    "data",

    "MAMI_2022_images",

    "training_images"
)

TEST_IMAGE_DIR = os.path.join(

    path,

    "MAMI_Dataset",

    "data",

    "MAMI_2022_images",

    "test_images"
)

In [ ]:
print(os.path.exists(TRAIN_CSV))

print(os.path.exists(DEV_CSV))

print(os.path.exists(TEST_CSV))

print(os.path.exists(TRAIN_IMAGE_DIR))

print(os.path.exists(TEST_IMAGE_DIR))

In [ ]:
train_df = pd.read_csv(
    TRAIN_CSV,
    sep="\t"
)

dev_df = pd.read_csv(
    DEV_CSV,
    sep="\t"
)

test_df = pd.read_csv(
    TEST_CSV,
    sep="\t"
)

In [ ]:
print(train_df.head())

print(train_df.columns)

In [ ]:
train_df = train_df[[

    "file_name",

    "text",

    "label"
]]

dev_df = dev_df[[

    "file_name",

    "text",

    "label"
]]

test_df = test_df[[

    "file_name",

    "text",

    "label"
]]

In [ ]:
train_df["label"] = (

    train_df["label"]

    .astype(str)

    .str.strip()

    .str.lower()
)

dev_df["label"] = (

    dev_df["label"]

    .astype(str)

    .str.strip()

    .str.lower()
)

test_df["label"] = (

    test_df["label"]

    .astype(str)

    .str.strip()

    .str.lower()
)

In [ ]:
label_map = {

    "misogynous": 1,

    "non-misogynous": 0,

    "1": 1,

    "0": 0
}

In [ ]:
train_df["label"] = train_df["label"].map(
    label_map
)

dev_df["label"] = dev_df["label"].map(
    label_map
)

test_df["label"] = test_df["label"].map(
    label_map
)

In [ ]:
train_df = train_df.dropna(
    subset=["label"]
)

dev_df = dev_df.dropna(
    subset=["label"]
)

test_df = test_df.dropna(
    subset=["label"]
)

In [ ]:
train_df["label"] = train_df["label"].astype(int)

dev_df["label"] = dev_df["label"].astype(int)

test_df["label"] = test_df["label"].astype(int)

In [ ]:
train_df = train_df.head(1000)

dev_df = dev_df.head(500)

test_df = test_df.head(500)

In [ ]:
def filter_existing_images(

    df,

    image_dir
):

    valid_rows = []

    for idx in range(len(df)):

        row = df.iloc[idx]

        image_path = os.path.join(

            image_dir,

            str(row["file_name"]).strip()
        )

        if os.path.exists(image_path):

            valid_rows.append(row)

    return pd.DataFrame(valid_rows)

In [ ]:
train_df = filter_existing_images(

    train_df,

    TRAIN_IMAGE_DIR
)

dev_df = filter_existing_images(

    dev_df,

    TRAIN_IMAGE_DIR
)

test_df = filter_existing_images(

    test_df,

    TEST_IMAGE_DIR
)

In [ ]:
print(len(train_df))

print(len(dev_df))

print(len(test_df))

In [ ]:
train_data = train_df.to_dict(
    "records"
)

dev_data = dev_df.to_dict(
    "records"
)

test_data = test_df.to_dict(
    "records"
)

In [ ]:
def convert_dataset(

    data,

    image_dir
):

    converted = []

    for sample in data:

        image_path = os.path.join(

            image_dir,

            sample["file_name"]
        )

        answer = (

            "Yes"

            if sample["label"] == 1

            else "No"
        )

        converted.append({

            "image": image_path,

            "question": f'''
            Meme Text:
            {sample["text"]}

            Is this meme misogynous?

            Answer Yes or No.
            ''',

            "answer": answer,

            "label": sample["label"]
        })

    return converted

In [ ]:
train_converted = convert_dataset(

    train_data,

    TRAIN_IMAGE_DIR
)

dev_converted = convert_dataset(

    dev_data,

    TRAIN_IMAGE_DIR
)

test_converted = convert_dataset(

    test_data,

    TEST_IMAGE_DIR
)

In [ ]:
train_dataset = Dataset.from_list(
    train_converted
)

dev_dataset = Dataset.from_list(
    dev_converted
)

test_dataset = Dataset.from_list(
    test_converted
)

In [ ]:
MODEL_ID = "Salesforce/instructblip-flan-t5-xl"

In [ ]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [ ]:
processor = InstructBlipProcessor.from_pretrained(
    MODEL_ID
)

In [ ]:
model = InstructBlipForConditionalGeneration.from_pretrained(

    MODEL_ID,

    quantization_config=bnb_config,

    device_map="auto"
)

In [ ]:
model = prepare_model_for_kbit_training(
    model
)

In [ ]:
lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[

        "query",

        "key",

        "value"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="SEQ_2_SEQ_LM"
)

In [ ]:
model = get_peft_model(

    model,

    lora_config
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
def preprocess_function(example):

    image = Image.open(

        example["image"]

    ).convert("RGB")

    model_inputs = processor(

        images=image,

        text=example["question"],

        return_tensors="pt",

        padding="max_length",

        truncation=True,

        max_length=128
    )

    labels = processor.tokenizer(

        example["answer"],

        padding="max_length",

        truncation=True,

        max_length=10,

        return_tensors="pt"
    )

    processed = {

        "pixel_values": model_inputs["pixel_values"].squeeze(0),

        "input_ids": model_inputs["input_ids"].squeeze(0),

        "attention_mask": model_inputs["attention_mask"].squeeze(0),

        "qformer_input_ids": model_inputs["qformer_input_ids"].squeeze(0),

        "qformer_attention_mask": model_inputs["qformer_attention_mask"].squeeze(0),

        "labels": labels["input_ids"].squeeze(0)
    }

    return processed

In [ ]:
train_dataset = train_dataset.map(
    preprocess_function
)

dev_dataset = dev_dataset.map(
    preprocess_function
)

In [ ]:
columns_to_keep = [

    "pixel_values",

    "input_ids",

    "attention_mask",

    "qformer_input_ids",

    "qformer_attention_mask",

    "labels"
]

In [ ]:
train_dataset = train_dataset.remove_columns(

    [

        col

        for col in train_dataset.column_names

        if col not in columns_to_keep
    ]
)

dev_dataset = dev_dataset.remove_columns(

    [

        col

        for col in dev_dataset.column_names

        if col not in columns_to_keep
    ]
)

In [ ]:
training_args = TrainingArguments(

    output_dir="./instructblip_mami",

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    num_train_epochs=10,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=True,

    remove_unused_columns=False,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset
)

In [ ]:
start_train_time = time.time()

trainer.train()

end_train_time = time.time()

training_time = (

    end_train_time -

    start_train_time
)

print(training_time)

In [ ]:
#trainer.save_model(
    #"./instructblip_mami_model"
#)

In [ ]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

In [ ]:
def predict_instructblip(

    image_path,

    text
):

    image = Image.open(

        image_path

    ).convert("RGB")

    prompt = f"""
    Meme Text:
    {text}

    Is this meme misogynous?

    Answer Yes or No.
    """

    inputs = processor(

        images=image,

        text=prompt,

        return_tensors="pt"

    ).to(device)

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=10
        )

    result = processor.batch_decode(

        outputs,

        skip_special_tokens=True

    )[0]

    result = result.lower()

    if "yes" in result:

        return 1

    else:

        return 0

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:

    try:

        from pynvml import *

        nvmlInit()

        handle = nvmlDeviceGetHandleByIndex(0)

        ENERGY_MODE = "GPU"

    except:

        ENERGY_MODE = "CPU"

else:

    ENERGY_MODE = "CPU"

In [ ]:
def get_power_usage():

    if ENERGY_MODE == "GPU":

        return (

            nvmlDeviceGetPowerUsage(handle)

            / 1000
        )

    else:

        return 65

In [ ]:
def compute_energy_kwh(

    power_watts,

    duration_seconds
):

    return (

        power_watts *

        duration_seconds

    ) / (1000 * 3600)

In [ ]:
y_true = []

y_pred = []

y_scores = []

inference_times = []

energy_consumptions = []

In [ ]:
for sample in tqdm(test_data):

    image_path = os.path.join(

        TEST_IMAGE_DIR,

        sample["file_name"]
    )

    text = sample["text"]

    label = int(sample["label"])

    start_time = time.time()

    power = get_power_usage()

    prediction = predict_instructblip(

        image_path,

        text
    )

    end_time = time.time()

    duration = end_time - start_time

    energy = compute_energy_kwh(

        power,

        duration
    )

    y_true.append(label)

    y_pred.append(int(prediction))

    y_scores.append(int(prediction))

    inference_times.append(duration)

    energy_consumptions.append(energy)

In [ ]:
accuracy = accuracy_score(

    y_true,

    y_pred
)

precision = precision_score(

    y_true,

    y_pred
)

recall = recall_score(

    y_true,

    y_pred
)

f1 = f1_score(

    y_true,

    y_pred
)

avg_time = np.mean(
    inference_times
)

avg_energy = np.mean(
    energy_consumptions
)

In [ ]:
print("ACCURACY:", accuracy)

print("PRECISION:", precision)

print("RECALL:", recall)

print("F1-SCORE:", f1)

print("AVERAGE INFERENCE TIME (s):", avg_time)

print("AVERAGE ENERGY (kWh):", avg_energy)

In [ ]:
report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Misogynous",

        "Misogynous"
    ],

    digits=4
)

print("\nCLASSIFICATION REPORT:\n")

print(report)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

labels = [

    "Non-Misogynous",

    "Misogynous"
]

plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.colorbar()

plt.xticks(
    [0,1],
    labels
)

plt.yticks(
    [0,1],
    labels
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(

    y_true,

    y_scores
)

roc_auc = auc(

    fpr,

    tpr
)

plt.figure(figsize=(6,5))

plt.plot(

    fpr,

    tpr,

    label=f"AUC = {roc_auc:.4f}"
)

plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
results_df = pd.DataFrame({

    "True_Label": y_true,

    "Predicted_Label": y_pred,

    "Inference_Time": inference_times,

    "Energy_kWh": energy_consumptions
})

results_df.to_csv(

    "instructblip_mami_results.csv",

    index=False
)

In [ ]:
print(results_df.head())